## AuxTel test LTS-337-003 (Nasmyth rotator)

In this notebook, I slew the Nasmyth2 rotator through it's range of travel, and evaluate range, max slew speed, and accuracy of position.  Desired specs:

| Description | Value       | Unit          |   Name     |
| :---        |    :----:   |       :----:  |       ---: |
|The rotator shall have a minimum range of rotation of:    | ±120       | Degrees   |Aux_Tel_Field_Rotation_Range|
|The rotator shall be able to achieve or surpass this velocity during slews of the telescope.  |3.5       | Degrees/Second      |Aux_Tel_Inst_Rot_Max_Vel|
|The rotator shall have at maximum this absolute angle error.      | 0.01|Degrees|Aux_Tel_Inst_Rot_Abs_Error|

In [2]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from astropy.time import Time, TimeDelta
from lsst_efd_client import EfdClient


Bad key "text.kerning_factor" on line 4 in
/opt/lsst/software/stack/conda/miniconda3-py37_4.8.2/envs/lsst-scipipe-cb4e2dc/lib/python3.7/site-packages/matplotlib/mpl-data/stylelib/_classic_test_patch.mplstyle.
You probably need to get an updated matplotlibrc file from
http://github.com/matplotlib/matplotlib/blob/master/matplotlibrc.template
or from the matplotlib source distribution


In [ ]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [ ]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

In [ ]:
# Get EFD client and bring in Lupton's unpacking code
client = EfdClient('summit_efd')

def merge_packed_time_series(packed_dataframe, base_field, stride=1, 
                             ref_timestamp_col="cRIO_timestamp", internal_time_scale="tai"):
    """Select fields that are time samples and unpack them into a dataframe.
            Parameters
            ----------
            packedDF : `pandas.DataFrame`
                packed data frame containing the desired data
            base_field :  `str`
                Base field name that will be expanded to query all
                vector entries.
            stride : `int`, optional
                Only use every stride value when unpacking.  Must be a factor
                of the number of packed values.
                (1 by default)
            ref_timestamp_col : `str`, optional
                Name of the field name to use to assign timestamps to unpacked
                vector fields (default is 'cRIO_timestamp').
            internal_time_scale : `str`, optional
                Time scale to use when converting times to internal formats
                ('tai' by default). Equivalent to EfdClient.internal_scale
        Returns
            -------
            result : `pandas.DataFrame`
                A `pandas.DataFrame` containing the results of the query.
            """
    
    packed_fields = [k for k in packed_dataframe.keys() if k.startswith(base_field)]
    packed_fields = sorted(packed_fields, key=lambda k: int(k[len(base_field):]))  # sort by pack ID
    npack = len(packed_fields)
    if npack%stride != 0:
        raise RuntimeError(f"Stride must be a factor of the number of packed fields: {stride} v. {npack}")
    packed_len = len(packed_dataframe)
    n_used = npack//stride   # number of raw fields being used
    output = np.empty(n_used*packed_len)
    times = np.empty_like(output, dtype=packed_dataframe[ref_timestamp_col][0])
    
    if packed_len == 1:
        dt = 0
    else:
        dt = (packed_dataframe[ref_timestamp_col][1] - packed_dataframe[ref_timestamp_col][0])/npack
    for i in range(0, npack, stride):
        i0 = i//stride
        output[i0::n_used] = packed_dataframe[f"{base_field}{i}"]
        times[i0::n_used] = packed_dataframe[ref_timestamp_col] + i*dt
     
    timestamps = Time(times, format='unix', scale=internal_time_scale).datetime64
    return pd.DataFrame({base_field:output, "times":times}, index=timestamps)

In [ ]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

In [ ]:
# enable components if required
await atcs.enable()
#await latiss.enable()

In [ ]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [ ]:
# turn on ATAOS corrections just to make sure the mirror is under air
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=False)

In [ ]:
# Ensure we're using Nasmyth 2
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [ ]:
# slew telescope to desired starting position
# rotator does not move for this test as it's part of a different
# requirement/verification
start_az=0
start_el=80
start_rot_pa=0.0
await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)

In [ ]:
# Here is where the tests are carried out.
max_rot = 130.0 # Maximum +/- 130 degrees
rot_step = 10.0 # 10 degree steps
n_steps = int(max_rot / rot_step)
maxes = []
errors = []
speeds = []
signs = [-1.0, 1.0] # Repeat tests in both directions

for sign in signs:
    await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
    # Move to max_rot
    current_rot = start_rot_pa + sign * max_rot
    await atcs.point_azel(start_az, start_el, rot_tel=current_rot, wait_dome=False)
    current_time = datetime.now().strftime("%Y-%m-%dT%H:%M:%-S")
    # Back up 10 seconds and get angle data for 10 seconds before that
    t_end = current_time + TimeDelta(-10.0, format='sec', scale='tai')
    nsec = 10
    nasmyth_angle = await client.select_time_series("lsst.sal.ATMCS.mount_Nasmyth_Encoders", ['*'],
                                              t_end - TimeDelta(nsec, format='sec'), t_end)
    angle = merge_packed_time_series(nasmyth_angle, 'nasmyth2CalculatedAngle', stride=1)
    angleList = angle.values.tolist()
    velocity = (angleList[-1][0] - angleList[0][0]) / (angleList[-1][1] - angleList[0][1])
    print(f"Measured velocity = {velocity}")
    speeds.append(abs(velocity))
    # Get current position to verify it is beyond the max
    await asyncio.sleep(1)
    current_time = datetime.now().strftime("%Y-%m-%dT%H:%M:%-S")
    t_end = current_time
    nsec = 1
    nasmyth_angle = await client.select_time_series("lsst.sal.ATMCS.mount_Nasmyth_Encoders", ['*'],
                                              t_end - TimeDelta(nsec, format='sec'), t_end)
    angle = merge_packed_time_series(nasmyth_angle, 'nasmyth2CalculatedAngle', stride=1)
    angleList = angle.values.tolist()
    current_angle = angleList[-1][0]
    print(f"Current rotator angle = {current_angle}")
    maxes.append(abs(current_angle))
    # Now slew through a range of angles and evaluate the error
    for n in range(n_steps):
        current_rot = current_rot - sign * rot_step
        await atcs.point_azel(start_az, start_el, rot_tel=current_rot, wait_dome=False)
        # Get current position and compare to set point
        await asyncio.sleep(1)
        current_time = datetime.now().strftime("%Y-%m-%dT%H:%M:%-S")
        t_end = current_time
        nsec = 1
        nasmyth_angle = await client.select_time_series("lsst.sal.ATMCS.mount_Nasmyth_Encoders", ['*'],
                                                  t_end - TimeDelta(nsec, format='sec'), t_end)
        angle = merge_packed_time_series(nasmyth_angle, 'nasmyth2CalculatedAngle', stride=1)
        angleList = angle.values.tolist()
        current_angle = angleList[-1][0]
        error = current_angle - current_rot
        print(f"Current rotator angle = {current_angle}. Set point = {current_rot}.  Error = {error}")
        errors.append(abs(error))


In [ ]:
# Now check to see if the specs are met:
Aux_Tel_Field_Rotation_Range = 120.0
if min(maxes) > Aux_Tel_Field_Rotation_Range:
    print("Aux_Tel_Field_Rotation_Range passed.  Spec = {Aux_Tel_Field_Rotation_Range}. Measured = {min(maxes)} ")
else:
    print("Aux_Tel_Field_Rotation_Range failed!  Spec = {Aux_Tel_Field_Rotation_Range}. Measured = {min(maxes)} ")

Aux_Tel_Inst_Rot_Max_Vel = 3.5
if min(speeds) > Aux_Tel_Inst_Rot_Max_Vel:
    print("Aux_Tel_Inst_Rot_Max_Vel passed.  Spec = {Aux_Tel_Inst_Rot_Max_Vel}. Measured = {min(speeds)} ")
else:
    print("Aux_Tel_Inst_Rot_Max_Vel failed!  Spec = {Aux_Tel_Inst_Rot_Max_Vel}. Measured = {min(speeds)} ")

Aux_Tel_Inst_Rot_Abs_Error = 0.01
if max(errors) < Aux_Tel_Inst_Rot_Abs_Error:
    print("Aux_Tel_Inst_Rot_Abs_Error passed.  Spec = {Aux_Tel_Inst_Rot_Abs_Error}. Worst case error = {max(errors)} ")
else:
    print("Aux_Tel_Inst_Rot_Abs_Error failed!  Spec = {Aux_Tel_Inst_Rot_Abs_Error}. Worst case error = {max(errors)} ")


In [25]:
# take event checking out of the slew commands to test telescope only
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [26]:
# Now shut down
await atcs.shutdown()

Disabling ATAOS corrections
Cover state <MirrorCoverState.CLOSED: 6>
M1 cover already closed.
M1 vent state <VentsPosition.CLOSED: 1>
M1 vents already closed.
Close dome.
ATDome Shutter Door is already closed. Ignoring.
Slew dome to Park position.
Sending ATDomeTrajectory to DISABED state. Component will be left in DISABLEDstate or else it may send the ATDome back to alignment with the telescope.
process as completed...
atdometrajectory: <State.DISABLED: 1>
[Dome] delta Az = -082.940 deg
[Dome] delta Az = -082.820 deg
[Dome] delta Az = -082.460 deg
[Dome] delta Az = -081.870 deg
[Dome] delta Az = -081.030 deg
[Dome] delta Az = -079.970 deg
[Dome] delta Az = -078.680 deg
[Dome] delta Az = -077.160 deg
[Dome] delta Az = -075.390 deg
[Dome] delta Az = -073.390 deg
[Dome] delta Az = -071.150 deg
[Dome] delta Az = -068.700 deg
[Dome] delta Az = -066.000 deg
[Dome] delta Az = -063.070 deg
[Dome] delta Az = -059.930 deg
[Dome] delta Az = -056.560 deg
[Dome] delta Az = -053.110 deg
[Dome] delt